In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay
from matplotlib import pyplot as plt

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_SEXO','TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']

df = pd.read_parquet("enem_parquet", columns = colunas)

In [3]:
df = df.sample(n=1_000_000, random_state=42)

In [4]:
FALTOU = (
    (df['TP_PRESENCA_CH'] == 0) & 
    (df['TP_PRESENCA_LC'] == 0) & 
    (df['TP_PRESENCA_CN'] == 0) & 
    (df['TP_PRESENCA_MT'] == 0)
)

df['FALTOU'] = FALTOU.astype(int)

In [5]:
df = df[df['TP_ESCOLA'].isin([2,3])]
df = df[df['TP_ESTADO_CIVIL'].isin([1,2,3,4])]

In [6]:
df['TP_LOCALIZACAO_ESC'] = df['TP_LOCALIZACAO_ESC'].fillna(-1)
df['TP_SIT_FUNC_ESC'] = df['TP_SIT_FUNC_ESC'].fillna(-1)

In [7]:
df = df.dropna(subset=[f'Q{i:03d}' for i in range(1, 26)])

In [8]:
df = df.dropna(subset=[
    'NU_NOTA_CN',
    'NU_NOTA_CH',
    'NU_NOTA_LC',
    'NU_NOTA_MT'
])

In [9]:
df_reduzido = df[df['FALTOU'] == 0]

In [10]:
df_reduzido.isna().sum()

Q001                  0
Q002                  0
Q003                  0
Q004                  0
Q005                  0
Q006                  0
Q007                  0
Q008                  0
Q009                  0
Q010                  0
Q011                  0
Q012                  0
Q013                  0
Q014                  0
Q015                  0
Q016                  0
Q017                  0
Q018                  0
Q019                  0
Q020                  0
Q021                  0
Q022                  0
Q023                  0
Q024                  0
Q025                  0
TP_PRESENCA_LC        0
TP_PRESENCA_CH        0
TP_PRESENCA_CN        0
TP_PRESENCA_MT        0
TP_FAIXA_ETARIA       0
TP_SEXO               0
TP_ESTADO_CIVIL       0
TP_COR_RACA           0
TP_ESCOLA             0
TP_ST_CONCLUSAO       0
IN_TREINEIRO          0
NU_ANO                0
TP_LOCALIZACAO_ESC    0
TP_SIT_FUNC_ESC       0
NU_NOTA_CN            0
NU_NOTA_CH            0
NU_NOTA_LC      

In [11]:
df_reduzido['MEDIA'] = (df_reduzido['NU_NOTA_CN'] + df_reduzido['NU_NOTA_CH'] + df_reduzido['NU_NOTA_MT']+  df_reduzido['NU_NOTA_LC'] + df_reduzido['NU_NOTA_REDACAO']) / 5

# mediana = df_reduzido['MEDIA'].median()
# df_reduzido['CLASSE_NOTA'] = (df_reduzido['MEDIA'] < mediana).astype(int)

df_reduzido['CLASSE'] = df_reduzido.groupby('NU_ANO')['MEDIA'].transform(lambda x: pd.qcut(x, q=2, labels=[0,1])).astype('Int64')
    


df_reduzido['CLASSE'] = df_reduzido['CLASSE'].astype(int)

In [12]:
def transformar_colunas_ohe(df):
    
    colunas = [
        'Q001','Q002','Q003','Q004','Q006','Q007','Q008','Q009','Q010',
        'Q011','Q012','Q013','Q014','Q015','Q016','Q017','Q018',
        'Q019','Q020','Q021','Q022','Q023','Q024','Q025'
    ]
    
    df = df.dropna(subset=colunas)
    
    df = pd.get_dummies(df, columns=colunas, prefix=colunas, dtype=int)
    
    return df

In [13]:
df_reduzido = transformar_colunas_ohe(df_reduzido)

In [14]:
def agregar_questionario(df):
    df = df.copy()

    # Q001 — Escolaridade do PAI
    q001_cols = [f'Q001_{l}' for l in 'ABCDEFGH']
    df['escolaridade_pai'] = df[q001_cols].idxmax(axis=1).str.extract(r'_([A-H])')[0]
    df['escolaridade_pai'] = df['escolaridade_pai'].map(
        {'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7}
    )
    # Q002 — Escolaridade da MÃE
    q002_cols = [f'Q002_{l}' for l in 'ABCDEFGH']
    df['escolaridade_mae'] = df[q002_cols].idxmax(axis=1).str.extract(r'_([A-H])')[0]
    df['escolaridade_mae'] = df['escolaridade_mae'].map(
        {'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7}
    )

    # Escolaridade máxima entre os pais 
    df['escolaridade_pais_max'] = df[['escolaridade_pai','escolaridade_mae']].max(axis=1)

  
    # Q003 — Ocupação do PAI 
    # Q004 — Ocupação da MÃE
    q003_cols = [f'Q003_{l}' for l in 'ABCDEF']
    q004_cols = [f'Q004_{l}' for l in 'ABCDEF']
    df['ocupacao_pai'] = df[q003_cols].idxmax(axis=1).str.extract(r'_([A-F])')[0]
    df['ocupacao_pai'] = df['ocupacao_pai'].map({'A':0,'B':1,'C':2,'D':3,'E':4,'F':5})
    df['ocupacao_mae'] = df[q004_cols].idxmax(axis=1).str.extract(r'_([A-F])')[0]
    df['ocupacao_mae'] = df['ocupacao_mae'].map({'A':0,'B':1,'C':2,'D':3,'E':4,'F':5})

    # Q006 — Renda familiar
    q006_cols = [f'Q006_{l}' for l in 'ABCDEFGHIJKLMNOPQ']
    df['renda_familiar'] = df[q006_cols].idxmax(axis=1).str.extract(r'_([A-Q])')[0]
    df['renda_familiar'] = df['renda_familiar'].map(
        {l:i for i, l in enumerate('ABCDEFGHIJKLMNOPQ')}
    )

    # Q007/Q008 — Bens do domicílio 
    q007_cols = [f'Q007_{l}' for l in 'ABCD']
    q008_cols = [f'Q008_{l}' for l in 'ABCDE']
    df['score_bens_servicos'] = df[q007_cols].sum(axis=1)
    df['score_bens_dom']      = df[q008_cols].sum(axis=1)

    # Q009/Q010/Q011 — Equipamentos (TV, celular, computador, etc)
    
    q009_cols = [f'Q009_{l}' for l in 'ABCDE']
    q010_cols = [f'Q010_{l}' for l in 'ABCDE']
    q011_cols = [f'Q011_{l}' for l in 'ABCDE']
    df['score_equipamentos'] = (
        df[q009_cols].sum(axis=1) +
        df[q010_cols].sum(axis=1) +
        df[q011_cols].sum(axis=1)
    )

    # Q012/Q013/Q014/Q015/Q016/Q017 — Cômodos e estrutura da casa
    q012_cols = [f'Q012_{l}' for l in 'ABCDE']
    q013_cols = [f'Q013_{l}' for l in 'ABCDE']
    df['score_estrutura_casa'] = (
        df[q012_cols].sum(axis=1) +
        df[q013_cols].sum(axis=1)
    )

    # Q024/Q025 — Acesso a computador e internet (capital digital)
    q024_cols = [f'Q024_{l}' for l in 'ABCDE']
    q025_cols = [f'Q025_{l}' for l in 'AB']
    df['acesso_computador'] = df[q024_cols].idxmax(axis=1).str.extract(r'_([A-E])')[0]
    df['acesso_computador'] = df['acesso_computador'].map({'A':0,'B':1,'C':2,'D':3,'E':4})
    df['acesso_internet']   = df[q025_cols].idxmax(axis=1).str.extract(r'_([A-B])')[0]
    df['acesso_internet']   = df['acesso_internet'].map({'A':0,'B':1})

    # # SCORE SOCIOECONÔMICO GERAL (proxy de NSE)
    # df['nse_score'] = (
    #     df['renda_familiar'] * 2 +          # peso maior na renda
    #     df['escolaridade_pais_max'] * 1.5 + # capital cultural
    #     df['score_bens_dom'] +
    #     df['score_equipamentos'] +
    #     df['acesso_computador']
    # )

    return df

In [15]:
df_reduzido = agregar_questionario(df_reduzido)

In [16]:
colunas_q_originais = [c for c in df_reduzido.columns 
                       if c.startswith('Q') and '_' in c]

X = df_reduzido.drop(
    columns=['MEDIA', 'CLASSE', 'TP_SEXO', 'TP_COR_RACA',
             'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 
             'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'NU_ANO',
             'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC',
             'TP_PRESENCA_LC', 'TP_PRESENCA_CH',
             'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 'FALTOU']
    + colunas_q_originais  # remove TODAS as dummies Q de uma vez
)

y = df_reduzido['CLASSE']

In [17]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [18]:
def treinar_rf(x_train, y_train, x_test, y_test, n_estimators, max_depth, max_features, min_samples_split, min_samples_leaf):
    
    rf= RandomForestClassifier(
        n_estimators=n_estimators,        
        max_depth=max_depth,            
        max_features = max_features,     
        min_samples_split = min_samples_split,    
        min_samples_leaf = min_samples_leaf,      
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    rf.fit(x_train, y_train)

    y_pred_train = rf.predict(x_train)
    y_pred_test  = rf.predict(x_test)

    ein  = 1 - accuracy_score(y_train, y_pred_train)
    eout = 1 - accuracy_score(y_test,  y_pred_test)

    print(f"\nEin:  {ein:.4f}")
    print(f"Eout: {eout:.4f}")
    print(f"Gap:  {eout - ein:.4f}  {'overfitting' if eout - ein > 0.05 else 'ok'}")
    print("\n" + classification_report(y_test, y_pred_test))

    return rf

In [19]:
n_estimators=60
max_depth=30          
max_features='log2'    
min_samples_split=50
min_samples_leaf=10
    
rf = treinar_rf(x_train, y_train, x_test, y_test, n_estimators, max_depth, max_features, min_samples_split, min_samples_leaf)

imp = pd.Series(rf.feature_importances_, index=X.columns)
imp = imp.sort_values(ascending=False)

print(imp.head(20))
print(imp.head(20).sum())


Ein:  0.2834
Eout: 0.2991
Gap:  0.0157  ok

              precision    recall  f1-score   support

           0       0.69      0.74      0.71     21851
           1       0.72      0.66      0.69     21848

    accuracy                           0.70     43699
   macro avg       0.70      0.70      0.70     43699
weighted avg       0.70      0.70      0.70     43699

renda_familiar           0.267167
TP_ESCOLA                0.193976
acesso_computador        0.144453
ocupacao_mae             0.061067
escolaridade_pais_max    0.060383
TP_FAIXA_ETARIA          0.058851
ocupacao_pai             0.057085
escolaridade_mae         0.052493
escolaridade_pai         0.046220
Q005                     0.034955
acesso_internet          0.021272
TP_ESTADO_CIVIL          0.002077
TP_ST_CONCLUSAO          0.000000
IN_TREINEIRO             0.000000
score_bens_dom           0.000000
score_bens_servicos      0.000000
score_estrutura_casa     0.000000
score_equipamentos       0.000000
dtype: float64
1

In [20]:
print(X.columns.tolist())

['Q005', 'TP_FAIXA_ETARIA', 'TP_ESTADO_CIVIL', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 'escolaridade_pai', 'escolaridade_mae', 'escolaridade_pais_max', 'ocupacao_pai', 'ocupacao_mae', 'renda_familiar', 'score_bens_servicos', 'score_bens_dom', 'score_equipamentos', 'score_estrutura_casa', 'acesso_computador', 'acesso_internet']


In [21]:
y_prob = rf.predict_proba(x_test)[:,1]
roc_auc_score(y_test, y_prob)

0.7719368093526342